# Python, part 2 · Lists, Dictionaries and Loops

**Goal:** write the function that finds every "don't know" code in the file, which is the job of course 1's `apply_null_tokens`. Then read the rules inside `display_name`.

You'll learn lists, dictionaries, `for` loops and `if`. Those four ideas are behind most of the cleaning code you'll ever read.

**How to use this notebook** (the same as course 1)
- Start the lab cluster first (**Compute**; a cold start takes 5 minutes or more), then attach this notebook to it.
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = "P2"

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
import pandas as pd
from lnl import check, cleaning, inside, pipeline
from lnl.check import ___

## 1. How many records have no disciplinary points recorded?
Here's the file after part 0's first two steps: spaces trimmed, blanks made empty, rows labeled by offender number. Count the empty values in `disciplinary_points`, then look at what's actually in the column:

In [ ]:
raw = pipeline.load_raw()
step2 = cleaning.set_record_id(cleaning.strip_and_null(raw))

print("Empty:", step2["disciplinary_points"].isna().sum())
step2["disciplinary_points"].value_counts(dropna=False).head(8)

Counting empty values says 3,005. But another **2,846** records say `N/A`, the source system's way of writing "don't know". The real answer is 5,851, almost double. A report that only counted the empty values would be wrong, and nobody would notice.

Course 1 kept a list of these codes in a **data dictionary**, agreed with the data owner. Here it is:

In [ ]:
tokens = cleaning.NULL_TOKENS
tokens

That's a dictionary of lists. By the end of this notebook you'll be able to read it, and you'll have written the function that uses it.

## 2. Lists: several values in one
A **list** holds values in order, between square brackets and separated by commas:

In [ ]:
codes = ["U", "UNK"]
print(codes)
print(codes[0])        # the first one: Python counts from 0
print(codes[1])        # the second one

`in` asks whether a value is in the list. The answer is `True` or `False`, like `==` in part 1:

In [ ]:
print("UNK" in codes)
print("Unknown" in codes)

🧪 **Your turn #1.** How many codes are in `codes`? Use `len()`, the same as you used on a string in part 1.

In [ ]:
answer_1 = ___      # len() of codes
check.p2_ex1(answer_1)

You can add to a list with `.append`, and `" ".join(...)` sticks a list of strings together into one string, with a space between each. You'll see both inside `display_name` at the end:

In [ ]:
words = []
words.append("De")
words.append("la")
words.append("Cruz")
print(words)
print(" ".join(words))

## 3. Dictionaries: look a value up by its name
A **dictionary** pairs each name (a *key*) with a value, between curly brackets: `{"key": value, ...}`. Look a value up with the key in square brackets:

In [ ]:
print(tokens["escape_risk"])
print(tokens["protective_custody"])

Each value in this dictionary is a list: the codes that mean "don't know" in that column.

Notice what is **not** in the dictionary: `housing_restriction`. In that column `NONE` means *"no restriction needed"*, which is a real answer. Deciding what's a code and what's an answer is a job for the data owner, not a guess.

In [ ]:
print("housing_restriction" in tokens)

🧪 **Your turn #2.** Look up the codes for `intake_time`.

In [ ]:
answer_2 = ___      # tokens, with "intake_time" in square brackets
check.p2_ex2(answer_2)

## 4. Loops: do the same thing for every value
A `for` loop runs the indented lines once for each value in a list. Each time round, the name after `for` holds the next value:

In [ ]:
for code in codes:
    print("This code means don't know:", code)

Looping over a dictionary's `.items()` gives you each key and its value together:

In [ ]:
for column, column_codes in tokens.items():
    print(f"{column:20} {column_codes}")

## 5. `if`: decide what to do
`if` runs its indented lines only when the question is `True`. `else` runs when it's `False`:

In [ ]:
value = "UNK"
if value in codes:
    print(value, "means don't know")
else:
    print(value, "is a real value")

Python's own empty value is called `None`. When a function hands back `None`, it's saying "nothing here". That's what course 1 turns a "don't know" code into.

🧪 **Your turn #3.** Finish `clean_value`. When `value` is one of the `codes`, it should hand back `None`; otherwise, `value` just as it was. The check runs it on every different value in the four columns and compares it with course 1's `apply_null_tokens`.

In [ ]:
def clean_value(value, codes):
    """None (Python's empty value) when value is one of the codes; otherwise value, as it was."""
    unknown = ___      # True when value is in codes
    if unknown:
        return None
    return value

check.p2_ex3(clean_value)

## 6. The payoff: a loop inside a loop
Now use your function on the whole file. The outer loop goes through the four columns in the dictionary. For each one, the inner loop goes through every value in that column, about 97,000 of them, and counts the ones your function empties.

(If you haven't finished #3, these counts will be wrong. Finish it first.)

In [ ]:
for column, column_codes in tokens.items():
    count = 0
    for value in step2[column].dropna():
        if clean_value(value, column_codes) is None:
            count = count + 1
    print(f"{column:20} {count:6,} don't-know codes")

That was nearly 400,000 checks in a second or two. `count = count + 1` means "take `count`, add one, and keep the result under the same name". It's how you count in a loop.

The check below does the same and compares the result with course 1:

In [ ]:
check.p2_payoff(clean_value)

That 6,122 is the number from part 0's step 3. Add the 2,846 `N/A`s to the 3,005 empty values, and you have the real answer to section 1's question: **5,851**.

## 7. Read course 1's code
Here is how course 1 does the same job:

In [ ]:
inside.show(cleaning.apply_null_tokens)

| Line | Python | In plain English |
| --- | --- | --- |
| 4 | `for col, values in (tokens or NULL_TOKENS).items():` | For each column and its codes, from the dictionary it was given, or the course's own if it wasn't given one. |
| 5 | `if col in out.columns:` | Only if the table has that column. |
| 6 | `[v.upper() for v in values]` | A list made by a loop on one line: every code, in capitals. Capitalizing first is a good habit, in case a future file has `unk`. |
| 6 | `.str.upper().isin(...)` | `in`, asked of every value in the column at once. |
| 6 | `.mask(...)` | Put an empty value wherever the answer was `True`. |

Your loop did it one value at a time; pandas does it for a whole column at once. Part 3 is about that.

And here is `display_name`, with the `if` part 1 promised:

In [ ]:
inside.show(cleaning.display_name)

You can read almost all of it now:
- **Lines 6, 9, 11, 13 and 14:** an empty list, `.append`, and `" ".join(out)` at the end.
- **Line 7:** a `for` loop over the words of the name. `enumerate` also counts as it goes, so `i` is 0 for the first word, 1 for the second, and so on.
- **Lines 8 to 13:** `if`, `elif` ("else, if") and `else`. The first one that's `True` wins:
  - a suffix like `III` is looked up in the `NAME_SUFFIXES` dictionary (`.rstrip(".")` takes a dot off the end first, so `JR.` counts as `JR`);
  - a particle like `DE` becomes lowercase, but not when it's the first word (`i > 0`);
  - anything else gets the rules for *Mc* and apostrophes in `_cap_word`.
- **Lines 4 and 5:** `isinstance(name, str)` asks "is this a string?", and `not` turns `True` into `False` and back. So anything that isn't a string, such as an empty value, is handed straight back untouched.

## Wrap-up
- A **list** holds values in order (`codes[0]` is the first one). `in` asks whether a value is in it, `.append` adds to it, and `" ".join` sticks it together into one string.
- A **dictionary** looks a value up by its key: `tokens["intake_time"]`. Course 1's data dictionary is one.
- `for` does the same thing for every value, and `if` / `elif` / `else` decide what happens.
- `None` is Python's empty value.
- "Don't know" codes hide inside real-looking data. Counting only the empty values got section 1's answer wrong by half.

**Next, part 3 · pandas:** the same jobs on a whole column in one line, and you'll write course 1's `strip_and_null` yourself.

## Answers

**#1**
```python
answer_1 = len(codes)
```
**#2**
```python
answer_2 = tokens["intake_time"]
```
**#3**
```python
unknown = value in codes
```